# Laboratorio 3: Clasificación de Tapitas con CNN
## Zona de Inspección Industrial

### Objetivos
- Aplicar lo aprendido en Labs 1 y 2 a su proyecto real
- Configurar data augmentation **apropiado** para una zona de inspección fija
- Diseñar y entrenar una CNN para clasificar tapitas por color/forma
- Diagnosticar por qué un modelo puede fallar al probarlo con la cámara

### Contexto
En su proyecto, un **robot controlado por PLC** coloca tapitas en una **zona de inspección fija**.
Una cámara (celular) apunta a esa zona y una CNN debe clasificar la tapita.

> **La calidad de sus datos es MÁS importante que la complejidad de la CNN.**
> Un modelo simple con buenos datos supera a un modelo complejo con datos malos.

### Puntaje
| Actividad | Puntos |
|---|---|
| 1. Configuración y datos | 2 |
| 2. Data Augmentation | 3 |
| 3. CNN y entrenamiento | 3 |
| 4. Predicción | 2 |
| 5. Cámara y diagnóstico | 3 |
| **Total** | **13** |


In [ ]:
# Instalar dependencias
!pip install -q tensorflow matplotlib seaborn ipywidgets scikit-learn

In [ ]:
import os
import shutil
import random
import glob
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.metrics import classification_report, confusion_matrix

# Widgets interactivos
import ipywidgets as widgets
from IPython.display import display, clear_output

# Reproducibilidad
random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)

print(f"TensorFlow: {tf.__version__}")
print("Librerías cargadas correctamente")

---
## Actividad 1: Configuración y Preparación de Datos (2 pts)

### 1.1 Parámetros del proyecto
Complete los parámetros según **su** proyecto.


In [ ]:
# =============================================================
# ACTIVIDAD 1: Complete los parámetros de su proyecto
# =============================================================

IMG_SIZE =            # COMPLETAR: Tamaño de imagen en píxeles (recomendado: 128)
NUM_CLASSES =         # COMPLETAR: Número de clases
CLASS_NAMES = []      # COMPLETAR: Nombres EXACTOS de las carpetas en dataset/
                      # Ejemplo: ["rojo", "azul", "amarillo", "verde"]

BATCH_SIZE = 32
EPOCHS = 50
LEARNING_RATE =       # COMPLETAR: Tasa de aprendizaje (sugerencia: 0.001)

# Directorios
DATASET_DIR = '../data/tapitas'
OUTPUT_DIR = '../data/tapitas_split'

# --- Verificación automática ---
if os.path.exists(DATASET_DIR):
    carpetas = sorted([d for d in os.listdir(DATASET_DIR)
                       if os.path.isdir(os.path.join(DATASET_DIR, d))])
    print(f"Carpetas encontradas en {DATASET_DIR}/:")
    for c in carpetas:
        n = len([f for f in os.listdir(os.path.join(DATASET_DIR, c))
                 if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
        print(f"  {c}: {n} imágenes")
    if CLASS_NAMES and set(CLASS_NAMES) != set(carpetas):
        print(f"\n⚠ CLASS_NAMES={CLASS_NAMES} no coincide con las carpetas: {carpetas}")
else:
    print(f"⚠ No se encontró '{DATASET_DIR}'. Suba su dataset antes de continuar.")

### 1.2 División del Dataset

Se divide el dataset en **entrenamiento (80%)** y **validación (20%)**.

> **Importante para zona de inspección:** Los datos deben haberse capturado con el celular
> **en la misma posición y condiciones** donde estará durante la prueba real.
> Use `DividirVideo.py` con `interval_seconds >= 0.5` y `detect_stability = True`.


In [ ]:
def split_dataset(dataset_dir, output_dir, class_names, split_ratio=0.8):
    """Divide el dataset en train y validation."""
    train_dir = os.path.join(output_dir, 'train')
    val_dir = os.path.join(output_dir, 'validation')

    if os.path.exists(train_dir) and os.path.exists(val_dir):
        print(f"La división ya existe en '{output_dir}'. Usando datos existentes.")
        return

    print(f"Creando división en '{output_dir}'...")
    random.seed(42)

    for clase in class_names:
        clase_dir = os.path.join(dataset_dir, clase)
        if not os.path.exists(clase_dir):
            print(f"  ⚠ No se encontró: {clase_dir}")
            continue

        imgs = sorted([f for f in os.listdir(clase_dir)
                       if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
        random.shuffle(imgs)

        n_train = int(len(imgs) * split_ratio)

        for subset, img_list in [('train', imgs[:n_train]),
                                  ('validation', imgs[n_train:])]:
            dest = os.path.join(output_dir, subset, clase)
            os.makedirs(dest, exist_ok=True)
            for img in img_list:
                shutil.copy(os.path.join(clase_dir, img), os.path.join(dest, img))

        print(f"  {clase}: {n_train} train, {len(imgs) - n_train} validation")

    print("División completada.")

split_dataset(DATASET_DIR, OUTPUT_DIR, CLASS_NAMES)

### 1.3 Exploración Interactiva del Dataset

Use este widget para navegar las imágenes de cada clase.
Observe si las imágenes son **variadas** o muy **similares** entre sí.


In [ ]:
def explorar_dataset():
    """Widget interactivo para explorar el dataset."""
    clase_dd = widgets.Dropdown(options=CLASS_NAMES, description='Clase:')
    btn = widgets.Button(description='Mostrar otras', icon='refresh',
                         button_style='info')
    output = widgets.Output()

    def mostrar(_=None):
        with output:
            clear_output(wait=True)
            clase = clase_dd.value
            train_dir = os.path.join(OUTPUT_DIR, 'train', clase)

            if not os.path.exists(train_dir):
                print(f"No se encontró: {train_dir}")
                return

            imgs = [os.path.join(train_dir, f) for f in os.listdir(train_dir)
                    if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

            if not imgs:
                print("No hay imágenes en esta clase")
                return

            sample = random.sample(imgs, min(8, len(imgs)))
            fig, axes = plt.subplots(2, 4, figsize=(16, 8))
            for ax in axes.ravel():
                ax.axis('off')
            for ax, path in zip(axes.ravel(), sample):
                img = load_img(path, target_size=(IMG_SIZE, IMG_SIZE))
                ax.imshow(img)
                ax.set_title(os.path.basename(path)[:25], fontsize=8)

            n_total = len(imgs)
            val_dir = os.path.join(OUTPUT_DIR, 'validation', clase)
            n_val = len(os.listdir(val_dir)) if os.path.exists(val_dir) else 0
            plt.suptitle(f'Clase: {clase} — {n_total} train / {n_val} val',
                         fontsize=14)
            plt.tight_layout()
            plt.show()

    clase_dd.observe(lambda c: mostrar(), names='value')
    btn.on_click(mostrar)
    display(widgets.HBox([clase_dd, btn]))
    display(output)
    mostrar()

explorar_dataset()

### 1.4 ¿Importa la resolución?

Compare cómo se ve la misma imagen a diferentes tamaños.
¿A qué resolución se distinguen bien los **colores y formas** de las tapitas?


In [ ]:
def comparar_resoluciones():
    """Muestra la misma imagen a diferentes resoluciones."""
    ejemplo_dir = os.path.join(OUTPUT_DIR, 'train', CLASS_NAMES[0])
    imgs = [f for f in os.listdir(ejemplo_dir)
            if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    if not imgs:
        print("No hay imágenes disponibles")
        return
    ejemplo_path = os.path.join(ejemplo_dir, imgs[0])

    sizes = [32, 64, 128, 224]
    fig, axes = plt.subplots(1, 4, figsize=(16, 4))

    for ax, size in zip(axes, sizes):
        img = load_img(ejemplo_path, target_size=(size, size))
        ax.imshow(img)
        marca = " ← recomendado" if size == 128 else ""
        ax.set_title(f'{size}×{size} px{marca}', fontsize=12,
                     fontweight='bold' if size == 128 else 'normal')
        ax.axis('off')

    plt.suptitle('¿A qué resolución se distinguen los colores/formas?', fontsize=14)
    plt.tight_layout()
    plt.show()

comparar_resoluciones()

**Pregunta 1.1:** ¿Las imágenes dentro de cada clase se ven muy similares entre sí?
¿Por qué podría ser un problema para la generalización del modelo?

**Respuesta:** *(Escriba aquí)*

**Pregunta 1.2:** ¿A qué resolución se distinguen bien los colores de las tapitas?
¿Por qué no usar 224×224 siempre?

**Respuesta:** *(Escriba aquí)*


---
## Actividad 2: Data Augmentation para Zona de Inspección (3 pts)

En los Labs 1 y 2 usamos `ImageDataGenerator` para crear variaciones de las imágenes.

Pero **no todas las augmentations son útiles para todos los problemas**.
Para una zona de inspección fija, debemos pensar: *¿qué variaciones ocurren realmente?*

| Variación real | Augmentation correspondiente |
|---|---|
| La tapita cae rotada | `rotation_range` |
| El robot no coloca en el punto exacto | `width/height_shift_range` (pequeño) |
| La luz del lab cambia entre sesiones | `brightness_range` |
| La cámara se mueve ligeramente | `zoom_range` (pequeño) |
| ¿La imagen se voltea en la práctica? | `horizontal_flip` → **probablemente NO** |

### 2.1 Experimente con los sliders

Mueva cada slider y observe el efecto. Piense cuáles son **relevantes para su caso**.


In [ ]:
def preview_augmentation():
    """Widget interactivo para previsualizar Data Augmentation."""
    # Cargar imagen de ejemplo
    ejemplo_dir = os.path.join(OUTPUT_DIR, 'train', CLASS_NAMES[0])
    imgs = [f for f in os.listdir(ejemplo_dir)
            if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    if not imgs:
        print("No hay imágenes disponibles")
        return
    ejemplo_path = os.path.join(ejemplo_dir, imgs[0])
    ejemplo_array = img_to_array(load_img(ejemplo_path,
                                          target_size=(IMG_SIZE, IMG_SIZE))) / 255.0

    # Selector de imagen
    clase_dd = widgets.Dropdown(options=CLASS_NAMES, description='Clase:')

    # Sliders
    sw = {'description_width': '120px'}
    rotacion  = widgets.IntSlider(min=0, max=90, step=5, value=0,
                                  description='Rotación (°):', style=sw)
    shift_h   = widgets.FloatSlider(min=0, max=0.3, step=0.05, value=0,
                                    description='Desplaz. Horiz:', style=sw)
    shift_v   = widgets.FloatSlider(min=0, max=0.3, step=0.05, value=0,
                                    description='Desplaz. Vert:', style=sw)
    brillo_min = widgets.FloatSlider(min=0.5, max=1.0, step=0.05, value=1.0,
                                     description='Brillo mín:', style=sw)
    brillo_max = widgets.FloatSlider(min=1.0, max=1.5, step=0.05, value=1.0,
                                     description='Brillo máx:', style=sw)
    zoom      = widgets.FloatSlider(min=0, max=0.3, step=0.05, value=0,
                                    description='Zoom:', style=sw)
    flip_h    = widgets.Checkbox(value=False, description='Flip horizontal')

    output = widgets.Output()
    img_ref = [ejemplo_array]  # Mutable reference para actualizar imagen

    def cambiar_clase(change):
        d = os.path.join(OUTPUT_DIR, 'train', change['new'])
        fs = [f for f in os.listdir(d) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        if fs:
            img_ref[0] = img_to_array(load_img(os.path.join(d, fs[0]),
                                                target_size=(IMG_SIZE, IMG_SIZE))) / 255.0
            actualizar()

    def actualizar(_=None):
        with output:
            clear_output(wait=True)
            datagen = ImageDataGenerator(
                rotation_range=rotacion.value,
                width_shift_range=shift_h.value,
                height_shift_range=shift_v.value,
                brightness_range=[brillo_min.value, brillo_max.value],
                zoom_range=zoom.value,
                horizontal_flip=flip_h.value,
                fill_mode='nearest'
            )
            fig, axes = plt.subplots(1, 6, figsize=(20, 3.5))
            axes[0].imshow(np.clip(img_ref[0], 0, 1))
            axes[0].set_title("ORIGINAL", fontweight='bold', color='blue')
            axes[0].axis('off')

            batch = np.expand_dims(img_ref[0], 0)
            gen = datagen.flow(batch, batch_size=1)
            for i in range(1, 6):
                aug = next(gen)[0]
                axes[i].imshow(np.clip(aug, 0, 1))
                axes[i].set_title(f"Variación {i}")
                axes[i].axis('off')
            plt.tight_layout()
            plt.show()

    for w in [rotacion, shift_h, shift_v, brillo_min, brillo_max, zoom, flip_h]:
        w.observe(actualizar, names='value')
    clase_dd.observe(cambiar_clase, names='value')

    sliders = widgets.VBox([clase_dd, rotacion, shift_h, shift_v,
                            brillo_min, brillo_max, zoom, flip_h])
    display(widgets.HBox([sliders, output]))
    actualizar()

preview_augmentation()

### 2.2 Preguntas sobre Data Augmentation

Piense en **su zona de inspección** y responda:

**a)** ¿Tiene sentido usar `rotation_range`? ¿Las tapitas pueden caer en cualquier orientación?

**Respuesta:** *(Escriba aquí)*

**b)** ¿Tiene sentido usar `horizontal_flip`? ¿La imagen se vería volteada en la práctica?

**Respuesta:** *(Escriba aquí)*

**c)** ¿Qué augmentation es la MÁS importante para que el modelo funcione en diferentes
sesiones del laboratorio? (Pista: ¿qué cambia entre una sesión y otra?)

**Respuesta:** *(Escriba aquí)*

**d)** ¿Por qué usamos desplazamientos **pequeños** (0.05-0.1) y no grandes (0.3+)?

**Respuesta:** *(Escriba aquí)*


### 2.3 Configure su ImageDataGenerator

Basándose en sus respuestas, elija los valores que considere apropiados para **su** zona de inspección.
Justifique brevemente cada elección.


In [ ]:
# =============================================================
# ACTIVIDAD 2: Configure el Data Augmentation para su caso
# =============================================================

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=___,            # COMPLETAR (¿la tapita cae rotada?)
    width_shift_range=___,         # COMPLETAR (¿el robot es preciso?)
    height_shift_range=___,        # COMPLETAR
    brightness_range=[___, ___],   # COMPLETAR (¿la luz cambia entre sesiones?)
    zoom_range=___,                # COMPLETAR (¿la cámara se mueve?)
    # horizontal_flip=___,         # COMPLETAR: ¿aplica en su caso?
    fill_mode='nearest'
)

# IMPORTANTE: Validación SOLO normaliza. Nunca augmentation en validación.
val_datagen = ImageDataGenerator(rescale=1./255)

# Justificación (completar):
# rotation_range=___ porque ...
# width/height_shift=___ porque ...
# brightness_range=___ porque ...
# zoom_range=___ porque ...

print("ImageDataGenerator configurado")

In [ ]:
# Crear generadores de datos
train_generator = train_datagen.flow_from_directory(
    os.path.join(OUTPUT_DIR, 'train'),
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True
)

val_generator = val_datagen.flow_from_directory(
    os.path.join(OUTPUT_DIR, 'validation'),
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False   # No mezclar para evaluación correcta
)

print(f"\nClases detectadas: {train_generator.class_indices}")
print(f"Imágenes de entrenamiento: {train_generator.samples}")
print(f"Imágenes de validación: {val_generator.samples}")

### 2.4 Verifique su augmentation

Ejecute la siguiente celda para ver cómo se ven las imágenes con **su** configuración aplicada.


In [ ]:
# Visualizar un batch de imágenes augmentadas
sample_batch, sample_labels = next(train_generator)
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for i, ax in enumerate(axes.ravel()):
    if i < len(sample_batch):
        ax.imshow(sample_batch[i])
        label_idx = np.argmax(sample_labels[i])
        ax.set_title(CLASS_NAMES[label_idx], fontsize=12)
    ax.axis('off')
plt.suptitle('Imágenes de entrenamiento (con augmentation aplicado)', fontsize=14)
plt.tight_layout()
plt.show()

# Distribución de clases
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
train_counts = [len(os.listdir(os.path.join(OUTPUT_DIR, 'train', c))) for c in CLASS_NAMES]
val_counts   = [len(os.listdir(os.path.join(OUTPUT_DIR, 'validation', c))) for c in CLASS_NAMES]
ax1.bar(CLASS_NAMES, train_counts, color='steelblue', alpha=0.8)
ax1.set_title('Distribución - Entrenamiento')
ax1.set_ylabel('Imágenes')
ax2.bar(CLASS_NAMES, val_counts, color='coral', alpha=0.8)
ax2.set_title('Distribución - Validación')
plt.tight_layout()
plt.show()

---
## Actividad 3: Arquitectura CNN y Entrenamiento (3 pts)

### 3.1 Diseño de la CNN

En Labs 1 y 2 construimos CNNs con `Conv2D` y `MaxPooling2D`.
Ahora agregamos dos técnicas que mejoran el rendimiento:

| Capa | ¿Qué hace? | ¿Por qué la usamos? |
|---|---|---|
| `BatchNormalization` | Normaliza las activaciones entre capas | Entrenamiento más estable y rápido |
| `Dropout(p)` | Apaga `p%` de neuronas al azar durante entrenamiento | Reduce overfitting |
| `GlobalAveragePooling2D` | Promedia cada mapa de características en un solo valor | Alternativa a `Flatten`, con menos parámetros |

**Se proporciona el Bloque 1 como ejemplo. Complete los bloques restantes.**


In [ ]:
# =============================================================
# ACTIVIDAD 3: Complete la arquitectura de la CNN
# =============================================================

inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))

# === BLOQUE 1 (ejemplo) ===
# Conv2D: extrae 32 características con filtros 3×3
# BatchNormalization: normaliza para entrenar más rápido
# MaxPooling2D: reduce la imagen a la mitad
x = layers.Conv2D(32, 3, padding='same', activation='relu')(inputs)
x = layers.BatchNormalization()(x)
x = layers.MaxPooling2D(2)(x)

# === BLOQUE 2 ===
# COMPLETAR: Similar al Bloque 1 pero con 64 filtros
# Pregunta: ¿Por qué aumentamos los filtros?
# (las primeras capas detectan bordes, las profundas detectan formas complejas)


# === BLOQUE 3 ===
# COMPLETAR: Similar con 128 filtros


# === CLASIFICADOR ===
# GlobalAveragePooling2D promedia cada mapa de características.
# Resultado: vector de tamaño = nro de filtros de la última capa.
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.5)(x)  # Apaga 50% de neuronas → evita memorizar

# COMPLETAR: Capa de salida
# ¿Cuántas neuronas necesita la última capa? (pista: una por clase)
# ¿Qué activación para clasificación multiclase? (pista: Lab 1)
# outputs = layers.Dense(___, activation='___')(x)


model = models.Model(inputs=inputs, outputs=outputs)

model.compile(
    optimizer=Adam(learning_rate=LEARNING_RATE),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

In [ ]:
# Visualizar la arquitectura
try:
    from tensorflow.keras.utils import plot_model
    plot_model(model, show_shapes=True, show_layer_names=True,
               to_file='modelo_arquitectura.png', dpi=80)
    from IPython.display import Image as IPImage
    display(IPImage('modelo_arquitectura.png'))
except Exception:
    print("(No se pudo generar imagen de la arquitectura, esto es opcional)")

# Resumen de parámetros
total = model.count_params()
trainable = sum([tf.size(w).numpy() for w in model.trainable_weights])
print(f"\nParámetros totales: {total:,}")
print(f"Parámetros entrenables: {trainable:,}")

### 3.2 Entrenamiento

Usamos dos **callbacks**:
- **EarlyStopping**: detiene el entrenamiento si la validación no mejora por 10 épocas
- **ReduceLROnPlateau**: reduce el learning rate si se estanca


In [ ]:
callbacks = [
    EarlyStopping(patience=10, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(factor=0.5, patience=5, verbose=1)
]

print("Iniciando entrenamiento...")
print(f"  Épocas máximas: {EPOCHS}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Learning rate: {LEARNING_RATE}")
print()

history = model.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=val_generator,
    callbacks=callbacks,
    verbose=1
)

In [ ]:
# === Curvas de entrenamiento ===
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

epochs_ran = range(1, len(history.history['accuracy']) + 1)

ax1.plot(epochs_ran, history.history['accuracy'], 'b-', label='Entrenamiento', linewidth=2)
ax1.plot(epochs_ran, history.history['val_accuracy'], 'r-', label='Validación', linewidth=2)
ax1.set_title('Precisión (Accuracy)')
ax1.set_xlabel('Época')
ax1.set_ylabel('Precisión')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(epochs_ran, history.history['loss'], 'b-', label='Entrenamiento', linewidth=2)
ax2.plot(epochs_ran, history.history['val_loss'], 'r-', label='Validación', linewidth=2)
ax2.set_title('Pérdida (Loss)')
ax2.set_xlabel('Época')
ax2.set_ylabel('Pérdida')
ax2.legend()
ax2.grid(True, alpha=0.3)

# Indicador automático
train_acc = history.history['accuracy'][-1]
val_acc = history.history['val_accuracy'][-1]
gap = train_acc - val_acc

if gap > 0.15:
    titulo = f'⚠ Posible OVERFITTING (train={train_acc:.1%}, val={val_acc:.1%}, gap={gap:.1%})'
    color_t = 'red'
elif val_acc < 0.7:
    titulo = f'⚠ Precisión baja ({val_acc:.1%}). Revisar datos o arquitectura.'
    color_t = 'orange'
else:
    titulo = f'Entrenamiento OK (val: {val_acc:.1%})'
    color_t = 'green'

fig.suptitle(titulo, color=color_t, fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# === Evaluación: Matriz de Confusión ===
val_generator.reset()
y_pred_probs = model.predict(val_generator, verbose=1)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = val_generator.classes
class_labels = list(val_generator.class_indices.keys())

# Reporte de clasificación
print("\nReporte de Clasificación:")
print("=" * 60)
print(classification_report(y_true, y_pred, target_names=class_labels, zero_division=0))

# Evaluación numérica
val_loss, val_acc = model.evaluate(val_generator, verbose=0)
print(f"Precisión final: {val_acc:.1%}")
print(f"Pérdida final: {val_loss:.4f}")

# Matriz visual
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(max(6, NUM_CLASSES * 1.5), max(5, NUM_CLASSES * 1.2)))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_labels, yticklabels=class_labels)
plt.xlabel('Predicción')
plt.ylabel('Real')
plt.title('Matriz de Confusión')
plt.tight_layout()
plt.show()

# Análisis automático de errores
print("\nAnálisis de errores:")
for i, name in enumerate(class_labels):
    total = cm[i].sum()
    correct = cm[i][i]
    if total > 0:
        acc_i = correct / total
        if acc_i < 0.8:
            confused_with = class_labels[np.argmax(np.delete(cm[i], i))]
            print(f"  ⚠ '{name}': {acc_i:.0%} correcto. Se confunde más con '{confused_with}'")
        else:
            print(f"  ✓ '{name}': {acc_i:.0%} correcto")

### 3.3 Análisis de Resultados

**a)** Observe las curvas de entrenamiento. ¿Hay overfitting?
¿Cómo lo identifica en las gráficas?

**Respuesta:** *(Escriba aquí)*

**b)** Observe la matriz de confusión. ¿Hay alguna clase que el modelo confunda más?
¿A qué podría deberse?

**Respuesta:** *(Escriba aquí)*

**c)** ¿El modelo es óptimo? Mencione fortalezas y puntos de mejora basándose en las métricas.

**Respuesta:** *(Escriba aquí)*


---
## Actividad 4: Función de Predicción (2 pts)

Complete la función para predecir imágenes individuales.
Esta es la **misma lógica** que usará `Prueba_video.py` para clasificar en tiempo real.


In [ ]:
# =============================================================
# ACTIVIDAD 4: Complete la función de predicción
# =============================================================

def predecir_imagen(img_path, model, class_names, img_size):
    """
    Predice la clase de una imagen.

    Pasos:
      1. Cargar imagen y redimensionar a (img_size, img_size)
      2. Normalizar dividiendo entre 255
      3. Expandir dimensiones (el modelo espera un batch)
      4. Predecir y mostrar resultado con barras de confianza
    """
    # COMPLETAR las 3 líneas siguientes:
    img = load_img(img_path, target_size=(___, ___))     # COMPLETAR: tamaño correcto
    img_array = img_to_array(img) / ___                   # COMPLETAR: normalizar
    img_array = np.expand_dims(img_array, axis=0)

    # Predicción
    predictions = model.predict(img_array, verbose=0)[0]
    pred_class = np.argmax(predictions)
    confidence = predictions[pred_class]

    # Visualización
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4),
                                    gridspec_kw={'width_ratios': [1, 1.2]})

    # Imagen
    ax1.imshow(load_img(img_path, target_size=(img_size, img_size)))
    color = 'green' if confidence > 0.7 else 'orange'
    ax1.set_title(f'{class_names[pred_class]} ({confidence:.1%})',
                  fontsize=14, color=color, fontweight='bold')
    ax1.axis('off')

    # Barras de confianza por clase
    colors = ['#2ecc71' if i == pred_class else '#bdc3c7'
              for i in range(len(class_names))]
    bars = ax2.barh(class_names, predictions, color=colors)
    ax2.set_xlim(0, 1)
    ax2.set_xlabel('Confianza')
    for bar, val in zip(bars, predictions):
        ax2.text(bar.get_width() + 0.02,
                 bar.get_y() + bar.get_height() / 2,
                 f'{val:.1%}', va='center', fontsize=10)

    plt.tight_layout()
    plt.show()
    return class_names[pred_class], confidence

In [ ]:
# Probar con imágenes del set de validación
val_images = []
for ext in ('*.jpg', '*.jpeg', '*.png'):
    val_images += glob.glob(os.path.join(OUTPUT_DIR, 'validation', '*', ext))

if val_images:
    sample = random.sample(val_images, min(6, len(val_images)))
    aciertos = 0
    for img_path in sample:
        clase_real = os.path.basename(os.path.dirname(img_path))
        pred, conf = predecir_imagen(img_path, model, CLASS_NAMES, IMG_SIZE)
        ok = pred == clase_real
        aciertos += int(ok)
        simbolo = "✓" if ok else "✗"
        print(f"  {simbolo} Real: {clase_real} | Predicción: {pred} ({conf:.1%})")
        print()
    print(f"Aciertos: {aciertos}/{len(sample)}")
else:
    print("No se encontraron imágenes de validación")

---
## Actividad 5: Exportación y Prueba con Cámara (3 pts)

### 5.1 Exportar el modelo


In [ ]:
codigo_alumno = "COMPLETAR"   # COMPLETAR: su código de alumno

modelo_path = f"cnn_{codigo_alumno}_tapas.h5"
model.save(modelo_path)
print(f"Modelo guardado: {modelo_path}")
print(f"\n{'='*50}")
print("CONFIGURACIÓN PARA Prueba_video.py:")
print(f"{'='*50}")
print(f"  MODEL_PATH = \"{modelo_path}\"")
print(f"  IMG_SIZE = {IMG_SIZE}")
print(f"  CLASS_NAMES = {CLASS_NAMES}")
print(f"  CONFIDENCE_THRESHOLD = 0.6")

### 5.2 Diagnóstico del modelo

Ejecute esta herramienta **antes** de probar con la cámara para verificar que todo está en orden.


In [ ]:
def diagnostico():
    """Diagnóstico completo del modelo antes de probar con cámara."""
    print("=" * 60)
    print("  DIAGNÓSTICO DEL MODELO")
    print("=" * 60)

    # 1. Precisión
    val_generator.reset()
    loss, acc = model.evaluate(val_generator, verbose=0)
    print(f"\n1. PRECISIÓN EN VALIDACIÓN: {acc:.1%}")
    if acc < 0.7:
        print("   ⚠ BAJA — el modelo no aprendió bien.")
        print("   → ¿Tiene suficientes imágenes? ¿Las clases están balanceadas?")
    elif acc < 0.9:
        print("   → ACEPTABLE — puede funcionar pero hay margen de mejora.")
    else:
        print("   ✓ BUENA")

    # 2. Overfitting
    if hasattr(history, 'history'):
        t_acc = history.history['accuracy'][-1]
        v_acc = history.history['val_accuracy'][-1]
        gap = t_acc - v_acc
        print(f"\n2. OVERFITTING CHECK:")
        print(f"   Train: {t_acc:.1%} | Val: {v_acc:.1%} | Gap: {gap:.1%}")
        if gap > 0.15:
            print("   ⚠ HAY OVERFITTING — el modelo memorizó los datos.")
            print("   → Más data augmentation, más Dropout, o más datos variados.")
        else:
            print("   ✓ Sin overfitting significativo")

    # 3. Distribución de predicciones
    val_generator.reset()
    preds = model.predict(val_generator, verbose=0)
    pred_classes = np.argmax(preds, axis=1)
    print(f"\n3. DISTRIBUCIÓN DE PREDICCIONES:")
    for i, name in enumerate(CLASS_NAMES):
        n = np.sum(pred_classes == i)
        print(f"   {name}: {n} predicciones ({n/len(pred_classes):.1%})")
    unique_preds = len(np.unique(pred_classes))
    if unique_preds < NUM_CLASSES:
        print(f"\n   ⚠ Solo predice {unique_preds} de {NUM_CLASSES} clases!")
        print(f"   → El modelo no aprendió a distinguir todas las clases.")

    # 4. Confianza promedio
    max_confs = np.max(preds, axis=1)
    print(f"\n4. CONFIANZA PROMEDIO: {np.mean(max_confs):.1%}")
    low_conf = np.sum(max_confs < 0.6)
    if low_conf > 0:
        print(f"   ⚠ {low_conf} predicciones con confianza < 60%")

    # 5. Checklist
    print(f"\n5. CHECKLIST PARA CÁMARA EN VIVO:")
    print(f"   □ ¿El celular está FIJO en la misma posición que al grabar?")
    print(f"   □ ¿La iluminación es similar a cuando grabó?")
    print(f"   □ ¿La zona de inspección se ve igual?")
    print(f"   □ ¿Configuró IMG_SIZE = {IMG_SIZE} en Prueba_video.py?")
    print(f"   □ ¿Configuró CLASS_NAMES = {CLASS_NAMES} en Prueba_video.py?")
    print(f"   □ ¿Configuró MODEL_PATH correctamente en Prueba_video.py?")
    print("\n" + "=" * 60)

diagnostico()

### 5.3 Análisis interactivo de errores

Use este widget para revisar las imágenes donde el modelo **se equivoca**.
Esto le ayudará a entender qué mejorar.


In [ ]:
def analizar_errores():
    """Widget para revisar imágenes mal clasificadas."""
    val_generator.reset()
    preds = model.predict(val_generator, verbose=0)
    pred_classes = np.argmax(preds, axis=1)
    true_classes = val_generator.classes
    filenames = val_generator.filenames
    class_labels = list(val_generator.class_indices.keys())

    # Encontrar errores
    errores = []
    for i in range(len(true_classes)):
        if pred_classes[i] != true_classes[i]:
            errores.append({
                'file': filenames[i],
                'real': class_labels[true_classes[i]],
                'pred': class_labels[pred_classes[i]],
                'conf': preds[i][pred_classes[i]]
            })

    if not errores:
        print("✓ No hay errores en el set de validación.")
        return

    print(f"Se encontraron {len(errores)} errores de {len(true_classes)} imágenes")
    print()

    n_show = min(8, len(errores))
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    for ax in axes.ravel():
        ax.axis('off')

    for i, ax in enumerate(axes.ravel()):
        if i >= n_show:
            break
        err = errores[i]
        img_path = os.path.join(OUTPUT_DIR, 'validation', err['file'])
        if os.path.exists(img_path):
            img = load_img(img_path, target_size=(IMG_SIZE, IMG_SIZE))
            ax.imshow(img)
            ax.set_title(f"Real: {err['real']}\nPred: {err['pred']} ({err['conf']:.0%})",
                         fontsize=10, color='red')

    plt.suptitle('Imágenes MAL clasificadas — ¿Qué tienen en común?', fontsize=14)
    plt.tight_layout()
    plt.show()

analizar_errores()

### 5.4 Preguntas finales

**a)** Luego de probar con `Prueba_video.py`, ¿se obtuvo el resultado esperado?
Si no, ¿por qué cree que falla?

**Respuesta:** *(Escriba aquí)*

**b)** Si el modelo tiene buenas métricas en validación pero falla con la cámara,
¿cuál es la causa más probable?
(Pista: piense en las **condiciones de captura** de los datos)

**Respuesta:** *(Escriba aquí)*

**c)** Proponga al menos **2 mejoras** relacionadas con los **datos** y **1 mejora**
relacionada con el **modelo**.

**Respuesta:** *(Escriba aquí)*

**d)** ¿Cómo se podría integrar este modelo en un sistema industrial real?
¿Cómo se relaciona con la Industria 5.0?

**Respuesta:** *(Escriba aquí)*
